# Metropolis-Hastings for a Beta-Bernoulli Posterior

This notebook is the first code companion for the chapter's post-1950 computational Bayes section. It shows what it means to sample from a posterior distribution with a Markov chain rather than calculate everything in closed form.

## Historical problem

After the 1953 Metropolis paper and the 1970 Hastings generalisation, Bayesian inference was no longer limited to models where the posterior could be written down and integrated exactly. A Markov chain could be designed so that, after a burn-in period, its states behave like draws from the target posterior.

Here we deliberately choose a simple model where the exact posterior is known. That gives us a clean test: does the Metropolis-Hastings chain recover the correct posterior shape and posterior mean?

## Mathematical setup

We observe Bernoulli outcomes with unknown success probability $\theta$.

- Prior: $\theta \sim \mathrm{Beta}(\alpha, \beta)$
- Data: $y_i \mid \theta \sim \mathrm{Bernoulli}(\theta)$ independently
- Posterior: $\theta \mid y \sim \mathrm{Beta}(\alpha + s, \beta + n - s)$

where $s = \sum_i y_i$ is the number of successes.

We will ignore the exact posterior for the algorithm itself and recover it using a random-walk Metropolis-Hastings sampler on the logit scale.

In [ ]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.special import expit, logit
from scipy.stats import beta

ROOT = Path.cwd().resolve().parents[0]
SHARED = ROOT / "00_shared"
if str(SHARED) not in sys.path:
    sys.path.append(str(SHARED))

from plotting import save_fig, set_plot_style

set_plot_style()
rng = np.random.default_rng(42)

## Data

We use a simple binomial-style dataset: 72 successes out of 100 trials. This is not meant to be a sophisticated application. The point is to see the algorithm clearly.

We place a weakly informative $\mathrm{Beta}(2,2)$ prior on $\theta$.

In [ ]:
n = 100
s = 72
alpha_prior = 2
beta_prior = 2

alpha_post = alpha_prior + s
beta_post = beta_prior + (n - s)

exact_mean = alpha_post / (alpha_post + beta_post)
exact_var = (alpha_post * beta_post) / (
    (alpha_post + beta_post) ** 2 * (alpha_post + beta_post + 1)
)

print(f"Posterior: Beta({alpha_post}, {beta_post})")
print(f"Exact posterior mean: {exact_mean:.4f}")
print(f"Exact posterior sd: {np.sqrt(exact_var):.4f}")

## Why sample on the logit scale?

A naive random walk for $\theta$ can propose invalid values outside $(0,1)$. A standard fix is to work with

$$
\phi = \log\left(\frac{\theta}{1-\theta}\right), \qquad \theta = \frac{1}{1 + e^{-\phi}}.
$$

We run a Gaussian random walk on $\phi$ and then map back to $\theta$. The target density for $\phi$ is the posterior for $\theta$ multiplied by the Jacobian term $\theta(1-\theta)$.

In [ ]:
def log_posterior_phi(phi, alpha_post, beta_post):
    theta = expit(phi)
    if np.any((theta <= 0.0) | (theta >= 1.0)):
        return -np.inf

    # Beta density up to an additive constant plus Jacobian of the logit transform.
    return (
        (alpha_post - 1) * np.log(theta)
        + (beta_post - 1) * np.log(1.0 - theta)
        + np.log(theta)
        + np.log(1.0 - theta)
    )


def metropolis_hastings(
    initial_theta,
    alpha_post,
    beta_post,
    n_steps=25_000,
    proposal_sd=0.35,
    rng=None,
):
    if rng is None:
        rng = np.random.default_rng()

    phi = logit(initial_theta)
    samples_theta = np.empty(n_steps)
    accepted = 0

    current_logp = log_posterior_phi(phi, alpha_post, beta_post)

    for t in range(n_steps):
        proposal_phi = phi + rng.normal(0.0, proposal_sd)
        proposal_logp = log_posterior_phi(proposal_phi, alpha_post, beta_post)

        log_accept_ratio = proposal_logp - current_logp
        if np.log(rng.uniform()) < log_accept_ratio:
            phi = proposal_phi
            current_logp = proposal_logp
            accepted += 1

        samples_theta[t] = expit(phi)

    return samples_theta, accepted / n_steps


## Run the chain

In [ ]:
samples, acceptance_rate = metropolis_hastings(
    initial_theta=0.5,
    alpha_post=alpha_post,
    beta_post=beta_post,
    n_steps=25_000,
    proposal_sd=0.35,
    rng=rng,
)

burn_in = 3_000
posterior_samples = samples[burn_in:]

print(f"Acceptance rate: {acceptance_rate:.3f}")
print(f"Posterior mean from chain: {posterior_samples.mean():.4f}")
print(f"Exact posterior mean:      {exact_mean:.4f}")

## Diagnostics

The three most useful first diagnostics are:

- the chain trace,
- the histogram of sampled values,
- the running estimate of a posterior expectation.

For this notebook we track the posterior mean $E[\theta \mid y]$.

In [ ]:
grid = np.linspace(0.001, 0.999, 500)
exact_density = beta.pdf(grid, alpha_post, beta_post)

running_mean = np.cumsum(samples) / np.arange(1, len(samples) + 1)

fig, axes = plt.subplots(3, 1, figsize=(9, 11))

axes[0].plot(samples[:5_000], lw=0.8, color="#1f77b4")
axes[0].axhline(exact_mean, color="#d62728", ls="--", lw=1.5, label="Exact posterior mean")
axes[0].set_title("Metropolis-Hastings trace for the first 5,000 iterations")
axes[0].set_ylabel(r"$\theta$")
axes[0].legend()

axes[1].hist(posterior_samples, bins=40, density=True, alpha=0.65, color="#4c78a8", label="MH samples")
axes[1].plot(grid, exact_density, color="#f58518", lw=2.2, label="Exact Beta posterior")
axes[1].set_title("Sampled posterior versus exact posterior")
axes[1].set_xlabel(r"$\theta$")
axes[1].set_ylabel("Density")
axes[1].legend()

axes[2].plot(running_mean, color="#54a24b", lw=1.2, label="Running estimate")
axes[2].axhline(exact_mean, color="#d62728", ls="--", lw=1.5, label="Exact posterior mean")
axes[2].set_title("Convergence of the posterior mean estimate")
axes[2].set_xlabel("Iteration")
axes[2].set_ylabel(r"Running mean of $\theta$")
axes[2].legend()

fig.tight_layout()
save_fig(fig, Path("figs") / "mh_beta_bernoulli_diagnostics.png")
plt.show()

## A simple autocorrelation check

Independent Monte Carlo draws would have essentially no lag dependence. MCMC draws are not independent, so the chain carries serial correlation. That is the price paid for sampling from difficult targets with local moves.

In [ ]:
def autocorrelation(x, max_lag=40):
    x = np.asarray(x)
    x_centered = x - x.mean()
    denom = np.dot(x_centered, x_centered)
    acf = [1.0]
    for lag in range(1, max_lag + 1):
        num = np.dot(x_centered[:-lag], x_centered[lag:])
        acf.append(num / denom)
    return np.array(acf)


acf = autocorrelation(posterior_samples, max_lag=40)
lags = np.arange(len(acf))

fig, ax = plt.subplots(figsize=(8, 4))
ax.vlines(lags, 0, acf, color="#4c78a8", lw=2)
ax.axhline(0.0, color="black", lw=0.8)
ax.set_title("Autocorrelation of posterior samples")
ax.set_xlabel("Lag")
ax.set_ylabel("ACF")
fig.tight_layout()
save_fig(fig, Path("figs") / "mh_beta_bernoulli_acf.png")
plt.show()

## Interpretation

This example is deliberately modest, but it shows the core idea of post-1950 Bayesian computation:

- we define a target posterior,
- construct a Markov chain with that posterior as its stationary distribution,
- use long-run samples to approximate expectations, intervals, and densities.

Because the exact posterior is known here, we can verify that the sampled histogram and posterior mean converge to the right answer. In harder models, this exact check is no longer available, which is why diagnostics become much more important.

## References

- Metropolis et al. (1953), *Equation of State Calculations by Fast Computing Machines*.
- Hastings (1970), *Monte Carlo Sampling Methods Using Markov Chains and Their Applications*.
- Robert and Casella (2004), *Monte Carlo Statistical Methods*.